<a href="https://colab.research.google.com/github/DavidSenseman/BIO1173/blob/main/BIO1173_Class_04_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<!-- BIO1173_Class_04_4:Rev 1 -->

---------------------------
**COPYRIGHT NOTICE:** This Jupyterlab Notebook is a Derivative work of [Jeff Heaton](https://github.com/jeffheaton) licensed under the Apache License, Version 2.0 (the "License"); You may not use this file except in compliance with the License. You may obtain a copy of the License at

> [http://www.apache.org/licenses/LICENSE-2.0](http://www.apache.org/licenses/LICENSE-2.0)

Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.

------------------------

# **BIO 1173: Intro Computational Biology**

<!-- BIO3523_Class_01_5:Rev 1 -->

## **Module 4 : Getting Started with Hermes Agent**

* Instructor: [David Senseman](mailto:David.Senseman@utsa.edu), [Department of Biology, Health and the Environment](https://sciences.utsa.edu/bhe/), [UT San Antonio](https://www.utsa.edu/)

### Module Content

* Class_04_1 : Hermes Agent and your LXC Container
* Class_04_2 : Customize Hermes Agent
* Class_04_3 : Connecting Hermes Agent to Telegram
* **Class_04_4 : Selecting Models**

## Google Colab Instructions

Run next code cell to map this Colab lesson's folder /content/drive to your Google Drive. This will allow you keep a copy of this Colab notebook which is your course textbook.

In [ ]:
# @title **Run this cell first**
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    Colab = True
    print("Note: Using Google CoLab")
    !curl ipinfo.io
except:
    print("**WARNING**: Your GDrive is not mapped to /content/drive ")
    print("**WARNING**: Your work will not be saved!")
    Colab = False

You should see something _similar_ to the following output:
```text
Mounted at /content/drive
Note: Using Google CoLab
{
  "ip": "34.139.115.187",
  "hostname": "187.115.139.34.bc.googleusercontent.com",
  "city": "North Charleston",
  "region": "South Carolina",
  "country": "US",
  "loc": "32.8546,-79.9748",
  "org": "AS396982 Google LLC",
  "postal": "29415",
  "timezone": "America/New_York",
  "readme": "https://ipinfo.io/missingauth"
}
```

## **Get Course Information**

Run the code in the cell below to retrieve your BIO1173 Course Key and your port number.

In [ ]:
# @title **Run this Cell to Get Your Course Information**

import urllib.request
import time
URL = ("https://biologicslab.co/BIO1173/backend_code/send_student_info.py?v="
        + str(time.time()))
try:
  req = urllib.request.Request(
      URL, headers={"Cache-Control": "no-cache, no-store, must-revalidate"})
  with urllib.request.urlopen(req, timeout=20) as r:
    exec(r.read().decode("utf-8"))
except Exception as exc:
  print(f"\u26a0\ufe0f  Could not load the course-information cell: {exc}")
  print("    Check your internet connection and run this cell again.")


If the code is correct, you should see something _similar_ to the following output:
```text
Name: Gosset, William S.
myUTSA_ID: abc123
BIO1173_KEY: BIO1173-F26-01-3OC-SBX-LF4
Port number: 23100

To log into your LXC container use: ssh -p 23100 abc123@lucy.utsa.edu
```

## **Obtaining an OpenRouter API Key**

Before you continue with this lesson, you may want to watch a short tutorial on how to obtain an OpenRouter API Key.


In [ ]:
# @title **OpenRouter API Key**
from IPython.display import HTML
video_id = "2FOg5_n9Dho?si=eDcbzM_HYN-8SLH1"
HTML(f"""
<iframe width="560" height="315"
  src="https://www.youtube.com/embed/{video_id}"
  title="YouTube video player"
  frameborder="0"
  allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture"
  allowfullscreen
  referrerpolicy="strict-origin-when-cross-origin">
</iframe>
""")

# **Selecting Models for Your Hermes Agent**

Your Hermes Agent will not work unless it is connected to a Large Language Model (LLM). So far you have been using a free **local model** called **Nemotron 3.5 Lightning 30B A3B NVFP4**. The reason this is a _local model_ is because the computer system running the model is physically located in the same **BHE LXD Cluster Facility** as the computer systems that house your LXC container.

One important advantage of using a local model is that you don't have to pay to use it. However, there are 2 problems with using a local model.

First, the local model has to be **_local_**. The means that it has to connected to the same local LAN as the computer that is running your Hermes Agent. This is not a problem at the moment since you are working in a LXC container provided to you by the Deparment of Biology, Health and the Environment (BHE) as a student currently enrolled in BIO1173. However, at the end of the semester, your free LXC container disappears in order to make room for new incoming students. While you can setup a your own Hermes Agent on one of your computers, or in a VPS service like Hostinger (https://www.hostinger.com/applications/hermes-agent) you won't access to the local models running in the BHE LXD Cluster Facility.    

The second problem with using a local model, is that they are not as **powerful** as cloud-based models. Cloud-based models are more powerful because they have a huge number of parameters (over 1 trillon). Because of their size, they too big to fit within a "normal computer" and can only run on a massive inference cluster within a huge datacenter. Companies like Anthropic and OpenAI provide access their model using something called an **API Key**. Before you can connect your Hermes Agent to a cloud-model, you will need to purchase an API Key.     

## **API Key Charges**

When you use your API key, you are billed for "tokens in" and "tokens out". Different models have different costs, with the newest, most capable models--the so-called **Frontier Models**-- costing the most.

For example, on July 24, 2026, Anthropic's frontier model **Claude  Fable** costs \$10 for 1 million input tokens (\$10/M input tokens) and \$50 dollars for 1 million output token (\$50/M output tokens). You could easily spend \$1,000 each week using this model with your Hermes Agent.

Even for a trivial prompt like **"Hello Hermes"**, the token consumption in Hermes Agent is drastically different from a standard ChatGPT or OpenRouter playground session.

Because Hermes is an autonomous agent framework, every API call carries substantial system overhead (tool definitions, system instructions, memory files, and skill indexes) before the model processes your actual two-word query.

---

### **Estimated Token Breakdown for "Hello Hermes"**

#### **Input Tokens (~12,000 to 16,000 tokens**):
* **Tool Schemas (JSON/XML):** **~8,000 – 10,000 tokens**  
  Hermes passes full schemas for all registered native tools (filesystem operations, web browsing, terminal execution, memory updates, subagents, etc.) on every request so the LLM knows how to invoke them.
* **System Prompt & Base Rules:** **~2,000 – 3,000 tokens**  
  Contains core behavioral instructions, safety guardrails, output formatting standards, and instructions on how to structure tool calls.
* **Skills Index (`## Skills`):** **~1,000 – 3,500 tokens**  
  The agent loads an index of installed skill definitions so it knows what capabilities it can dynamically pull in.
* **Memory & User State (`SOUL.md`, `MEMORY.md`, `USER.md`):** **~500 – 1,500 tokens**  
  The agent's personality traits, saved memories across sessions, and user profile information.
* **Your Message (`"Hello Hermes"`):** **~3 – 5 tokens**

#### **Output Tokens (~10 to 50 tokens**):
* Since "Hello Hermes" does not require executing any tools or bash commands, the LLM will recognize this as a simple greeting and bypass tool calls.
* **Agent Response:** A brief greeting reply like *"Hello! How can I assist you today?"* takes roughly **15 to 30 tokens**.

---

In fact, your Hermes Agent requires a minimun **context size** of 65536000 tokens just to start using any model!


##**Less Expensive Cloud-Based Models**

While "top-tier" frontier models can be very expensive to use, there are a large number of cloud-based models that are capable of performing the various tasks your agent will need to perform in this course.

If you are careful, and select the best model for a particular task, you should not spend more than \$50 on your API charges for the semester. Indeed, one of the main goals of this course is to help you learn how to optimize your use of models with your AI agent.  



## **Get your OpenRouter API Key**

Below are the steps that you need to follow to obtain your API from from a company called **OpenRouter**.

---------
**Step 1:** Go to the OpenRouter web page: https://openrouter.ai/

---------

**Step 2:** Sign Up

![__](https://biologicslab.co/BIO1173/images/class_04/class_04_4_image05A.png)

-------------
**Step 3:** Click on `Get API Key`

![__](https://biologicslab.co/BIO1173/images/class_04/class_04_4_image06A.png)

-----------
**Step 4:** Create a new API key with a $10 credit limit

![__](https://biologicslab.co/BIO1173/images/class_04/class_04_4_image07A.png)

------------
**Step 5:** Make a Copy of Your API key

![__](https://biologicslab.co/BIO1173/images/class_04/class_04_4_image08A.png)

## **EXTREMELY IMPORTANT**

To save your API key must use a **text editor**, _not_ a wordprocessor. If you are using Windows, your text editor app is called **Notepad**; If you are using a Mac, your text editor is called **TextEdit**.

Be careful **not** to let anyone see your OpenRouter API key. If you think your API key has been compromised, immediately go to OpenRouter to the API Keys page and delete your key. You can then generate a new key without charge.

##**YouTube Tutorial - Changing the Hermes Agent Model**

Before you proceed, it is suggested that you watch a short video tutorial about changing your Hermes model.

###Generate Your SSH Command

To help you log into your LXC container, run the next cell to generate your ssh command.

In [ ]:
# @title Generate ssh command
from google.colab import userdata
import os

# Get Secrets
myUTSA_ID = userdata.get('myUTSA_ID')
os.environ['myUTSA_ID'] = myUTSA_ID
PORT_NUM = userdata.get('PORT_NUM')
os.environ['PORT_NUM'] = PORT_NUM

print(f"ssh -p {PORT_NUM} {myUTSA_ID}@lucy.utsa.edu")

## **Log Into Your LXC Container**

Use the ssh command that you generated above to log into your LXC container. Do **not** start your Hermes Agent yet.

Instead, type this command in your terminal.
```text
hermes model
```

The will bring up a selection menu of the large number of models that are currently available for your Hermes Agent. Unless your terminal window is very tall, you will only see a small portion of this list where is says:
```text
 → (●) custom (direct API)  ← currently active
```
Here is the complete list as of Oct 20, 2026:

```text
Select provider:
  ↑↓ navigate  ENTER/SPACE select  ESC cancel

   (○) Nous Portal (Everything your agent needs, 300+ models with bundled tool use)
   (○) Fireworks AI (OpenAI-compatible direct model API)
   (○) OpenRouter (Pay-per-use API aggregator)
   (○) Mixture of Agents (named presets; aggregator acts after reference models)
   (○) NovitaAI (Cloud: Model API, Agent Sandbox, GPU Cloud)
   (○) LM Studio (Local desktop app with built-in model server)
   (○) Anthropic (Claude models via API key or Claude Code)
   (○) OpenAI ▸ (ChatGPT/Codex subscription or direct OpenAI API)
   (○) Qwen ▸ (Qwen Cloud / DashScope, Coding Plan, Token Plan & Qwen CLI OAuth)
   (○) xAI Grok ▸ (Direct API or SuperGrok / Premium+ OAuth)
   (○) Xiaomi MiMo (MiMo-V2.5 and V2 models: pro, omni, flash)
   (○) Tencent Hy ▸ (Hy4 / Hy3 via TokenHub & TokenPlan)
   (○) NVIDIA NIM (Nemotron models via build.nvidia.com or local NIM)
   (○) GitHub Copilot ▸ (GitHub token API or copilot --acp process)
   (○) Hugging Face Inference Providers
   (○) Google AI Studio (Native Gemini API)
   (○) Google Vertex AI (Gemini via GCP; OAuth2 service account or ADC, GCP billing/quotas)
   (○) DeepSeek (V3, R1, coder, direct API)
   (○) Z.AI / GLM (Zhipu direct API)
   (○) Kimi / Moonshot ▸ (Coding Plan, Moonshot global & China endpoints)
   (○) StepFun Step Plan (Agent / coding models via Step Plan API)
   (○) MiniMax ▸ (Global, OAuth Coding Plan & China endpoints)
   (○) Ollama Cloud (Cloud-hosted open models, ollama.com)
   (○) Arcee AI (Trinity models, direct API)
   (○) GMI Cloud (Multi-model direct API)
   (○) Kilo Code (Kilo Gateway API)
   (○) OpenCode ▸ (Zen pay-as-you-go or Go subscription)
   (○) AWS Bedrock (Claude, Nova, Llama, DeepSeek; IAM or API key)
   (○) Azure Foundry (OpenAI-style or Anthropic-style endpoint, your Azure AI deployment)
   (○) Vercel AI Gateway (Multi-model aggregator)
   (○) Actual Computer - hosted inference via api.actual.inc, or local offline inference via model.base_url in config.yaml
   (○) CommandCode — 20+ models via OpenAI-compatible API
   (○) CommandCode — Claude models via Anthropic Messages API
 → (●) custom (direct API)  ← currently active
   (○) DeepInfra — 100+ open models, pay-per-use
   (○) Meta Muse Spark family (Meta Superintelligence Labs)
   (○) Nebius Token Factory — OpenAI-compatible inference
   (○) Ramp Router (router.com) — routes each request to the cheapest model that clears your quality bar
   (○) Upstage (Solar API)
   (○) Local models (run open models on this machine — no account or API key)
   (○) Custom endpoint (enter URL manually)
   (○) Reasoning effort for the current model...
   (○) Configure auxiliary models...
   (○) Leave unchanged
```
Use your up arrow to move the selection to **OpenRouter (Pay-per-use API aggregator** and press the space bar. You should now see this response:

```text

Get one at: https://openrouter.ai/keys

No OpenRouter API key configured.
OPENROUTER_API_KEY (or Enter to cancel):

```
Copy your API key from your text editor and paste it immediately after the colon **`:`**.

Your terminal should now look like this:


![__](https://biologicslab.co/BIO1173/images/class_04/class_04_4_image09A.png)

Press **`Enter/Return`**.

This will bring up the selection menu for the currently available models on OpenRouter including their token costs. Use your down arrow to pick the model called **anthropic/claude-haiku-5.5** and press the space bar.

```text
Select default model:
                                                     In        Out    Cache  $/Mtok
  ↑↓ navigate  ENTER/SPACE select  / search  ESC cancel  ← previous

   (○) anthropic/claude-fable-5.1                  $10.00     $50.00    $0.25
   (○) anthropic/claude-fable-5                    $10.00     $50.00    $1.00
   (○) anthropic/claude-opus-5.5                    $4.00     $20.00    $0.20
   (○) anthropic/claude-opus-5                      $5.00     $25.00    $0.50
   (○) anthropic/claude-opus-4.8                    $5.00     $25.00    $0.50
   (○) anthropic/claude-sonnet-5.5                  $2.00     $10.00    $0.10
   (○) anthropic/claude-sonnet-5                    $2.00     $10.00    $0.20
 → (●) anthropic/claude-haiku-5.5                   $0.10      $0.50    $0.01
   (○) anthropic/claude-haiku-4.5                   $1.00      $5.00    $0.10
   (○) openai/gpt-6-astra                          $10.00     $50.00    $1.00
   (○) openai/gpt-6-astra-pro                      $10.00     $50.00    $1.00
   (○) openai/gpt-6.1-sol                           $2.00     $10.00    $0.10
   (○) openai/gpt-6.1-sol-pro                       $2.00     $10.00    $0.10
```
You may, or may not, see the reasoning effort menu.

If you don't see the reasoning effort menu, but instead you are brought back to the OpenRouter selection menu, just hit **`Esc`**. After a short time, you should see your terminal window.

If you do see this reasoning selection menu, select **medium**.

```text

Select reasoning effort:
  ↑↓ navigate  ENTER/SPACE select  ESC cancel  ← previous

   (○) minimal
   (○) low
→  (●) medium
   (○) high
   (○) xhigh
   (○) max
   (○) ultra
   (○) Disable reasoning
   (○) Skip (keep current)

```
Use your up/down arrows to select **medium** and press **`Enter/Return`**. Medium effort is the default. If you need to change later, you can do so install the TUI.

This will return you to your teminal where you should now the following:

```text

Default model set to: anthropic/claude-haiku-5.5 (via OpenRouter)

Reasoning effort set to: high

```
You have successfully switched your local model to a cloud-based model. As of October 20, 2026, Anthropic is offering **Haiku 5.5** at a remarkly low price of only \$0.10 per million input tokens and \$0.50 per million output tokens. Compare this to Anthropics' most capable "frontier" model, **Fable 5.1** that costs \$10.00 per million input token and \$50.00 per million output tokens. It is very likely that this is an **introductory** price that Anthropic will increase at some time in the not too distant future. One of the most attractive features of OpenRouter is that you can easily switch your model without changing your API Key or paying any additional money.

Let's see what your Hermes Agent thinks about its "new brain"?





##**Start Hermes Agent**

Start your Hermes Agent TUI using this command:
```bash
hermes --tui
```
Set the session title using this command:
```text
/new BIO1173_Class_04_4

```
Say **Yes** when Hermes asks if you want to start a new session.

Send the following prompt to your Hermes Agent to find out more about your new, cloud-based model:

```text
Tell me what model you are using and what capabilites that it has. How does it compare to a current frontier model like Anthropic's claude/fable-5.1?

```
Here is Hermes output:

```text
┊ Current model: anthropic/claude-haiku-5.5 via OpenRouter, per this
    session's runtime metadata.
    
  Claude Haiku 5.5 capabilities (from Anthropic's platform docs):           
    • Input/output: text and images in, text out
    • Context window: 1M tokens
    • Max output: 128K tokens (300K on the Batch API, beta)
    • Thinking: adaptive, on by default, with an effort setting
    (default medium)
    • Speed: Anthropic's fastest model, built for classification,
    routing, extraction, and subagent work
    • Tools and caching: client-side tool calling and prompt caching
    are supported. Bedrock's card lists structured outputs as
    unsupported, so check OpenRouter's feature list before relying
    on them.
    • Knowledge cutoff: June 2026                                                     
    
    Claude Fable 5.1 for comparison (Anthropic's most capable
    generally available model)
    • Same 1M context and 128K max output       
    • Adaptive thinking is always on, with a default effort of
    high
    • Slower than Haiku, aimed at long-running agentic coding,
    multi-day autonomous work, and complex knowledge work
    • $10 per million input tokens and $50 per million output
    
    How they compare: Fable 5.1 is the stronger model for hard,
    multi-step, long-horizon work. Haiku 5.5 trades some of that
    depth for speed and lower cost, which suits high-volume or
    latency-sensitive jobs. I didn't find Haiku's per-token price in
    these results, so I can't give a cost ratio. I also haven't run
    either model on benchmarks myself, so treat this as a comparison
    of published specs and positioning, not measured performance.  

```
###**PROBLEMS?**

In the event that there is a **problem** with your OpenRouter API key, you will see an output _similar_ to this one:

```text
❯ Tell me what model you are using and what capabilites that it has. How does it compare to Anthropic's current frontier model claude/fable-5.1

┊  OpenRouter rejected your API key, so the model can't be reached.
Update it in Settings → Providers, or run hermes setup in a terminal.

Provider said: HTTP 401: User not found.

Your request was not processed. Send it again if you still want me
to carry it out.
```
For some reason your OpenRouter API key was not correct. This can happen if you used a word processor like MS Word instead of a text editor like NotePad (Windows) or TextEdit (Mac) which insert invisible characters into your API key.

##**Generate Session Summary**


####**Step 1: Generate Session Summary**
In order to grade this Colab lesson, give this final command to your Hermes Agent to generate a summary of your work. Copy the following text into your clipboard.
```text
/generate-session-summary
```
To paste this command into the text input of your Hermes Agent, use **`Cmd+V`** (Mac) or **`Ctrl+Shift+V`** (Windows) to generate a summary of your session.

Here is what your TUI should look like after you have pasted the prompt into the input text area:

![__](https://biologicslab.co/BIO1173/images/class_04/class_04_2_image02B.png)

Press **`Enter/Return`** to send the prompt to your Hermes Agent.

This command will cause your Hermes Agent to generate a summary of all of your prompts during the session.

----------

####**Step 2: Copy Session Summary to your Clipboard**

Copy the output generated by your Hermes Agent into your clipboard. There are slightly different ways to do this if you are using a MacBook or a Windows laptop.

----------------

>### **![__](https://biologicslab.co/BIO3523/images/class_01/mac_icon_small.png)MacOS INSTRUCTIONS:**

>When you are selecting text from the Hermes Agent TUI, using your trackpad/mouse, you must hold down two keys: **`fn+option`** while you select your text. The selected text should turn white.

----------------

>### **![__](https://biologicslab.co/BIO3523/images/class_01/pc_icon_small.png)WINDOWS INSTRUCTIONS:**

>When you are selecting text from the Hermes Agent TUI, using your trackpad/mouse, you must hold down three keys while you select your text: **`Shift+Ctrl+Alt`**. The selected text should turn white.

Whether you have a Windows laptop or a MacBook, your selected text should look like this:

![__](https://biologicslab.co/BIO1173/images/class_01/select_session_summary.png)

-------------

When you are selecting the text, make sure to include the initial square bracket **`[`**, as illustrated here:

```text
[{ "session_id": "20261018_113443_d1ad25","index": 1,.....
  ....
```
The selected text must also include the final closing square bracket **`]`** as illustrated here:
```text
  ...
slash-commands\nyou have available to help me in this environment."}]
```

Then press **`Ctrl+Shift+C`** to copy the selected text to your clipboard if you are using a Windows laptop or **`Cmd+C`** if you are using a MacBook.

-------------

####**Step 3: Paste Session Summary to the Generate Summay Code Cell**

Position your cursor/pointer immediately below this line in code cell below:
```text
# Paste your Session Summary below the next line
session_summary = r"""
```
Press **`Ctrl+V`** (Windows) or **`Cmd+V`** (MacBook) to paste your summary.  

In [ ]:
# @title Generate Session Summary

# Paste your Session Summary below the next line
session_summary = r"""



"""


# Display Session Summary
print("Session Summary:")
print(session_summary)


# **Electronic Submission**

When you run the code in the cell below, it will grade your Colab notebook and tell you your pending grade as it currently stands. You will be given the choice to either submit your notebook for final grading or the option to continue your work on one (or more) steps.

Grant Access to your Colab Secrets if you are asked to do so.

In [ ]:
# @title  Electronic Submission

import urllib.request
import ssl
import time

url = "https://biologicslab.co/BIO1173/validate_session.py?v=" + str(time.time())
ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE
req = urllib.request.Request(
    url,
    headers={
        "Cache-Control": "no-cache, no-store, must-revalidate",
        "Pragma": "no-cache",
        "Expires": "0"
    }
)
with urllib.request.urlopen(req, context=ctx) as r:
    exec(r.read().decode("utf-8"))
main()